# Univerzalna krivulja otpora kugle — Cd(Re)

**Poglavlje U14: Bezdimenzijski brojevi, dimenzijska analiza i sličnost**

Ovaj interaktivni prikaz materijalizira glavni rezultat dimenzijske analize: otpor kugle svodi se na jednu univerzalnu krivulju $C_d(Re)$. Mijenjanjem brzine, promjera i fluida radna se točka pomiče po istoj krivulji — bez obzira na veličinu i vrstu fluida.

## Cilj

Buckinghamova analiza pokazuje da problem otpora kugle ($F_D, \rho, v, D, \mu$) ovisi samo o dvije $\Pi$-grupe, pa je $C_d = f(Re)$. Prikaz omogućuje:

1. mijenjanje brzine strujanja $v$;
2. mijenjanje promjera kugle $D$;
3. izbor fluida preko kinematičke viskoznosti $\nu$ i gustoće $\rho$;
4. praćenje kako se radna točka $(Re, C_d)$ kreće po jednoj univerzalnoj krivulji i kolika je pripadna sila otpora.

## Pretpostavke modela

- glatka kruta kugla u ustaljenoj struji;
- nestlačivo strujanje ($Ma < 0{,}3$);
- $C_d(Re)$ prema Clift-Gauvinovoj korelaciji, valjanoj do otporne krize (oko $Re \approx 2\cdot 10^5$);
- karakteristična duljina je promjer $D$, čeona površina $A = \pi D^2/4$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Layout

plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 10

## Računski model

Dimenzijska analiza svodi pet veličina na dvije grupe:

$$\Pi_1 = \frac{F_D}{\rho v^2 D^2}, \qquad \Pi_2 = \frac{\rho v D}{\mu} = Re \quad\Rightarrow\quad C_d = f(Re).$$

Koeficijent otpora i sila:

$$C_d = \frac{F_D}{\tfrac{1}{2}\rho v^2 A}, \qquad F_D = C_d\,\tfrac{1}{2}\rho v^2 A, \qquad Re = \frac{vD}{\nu}.$$

Korelacija za glatku kuglu (Clift-Gauvin):

$$C_d = \frac{24}{Re}\left(1 + 0{,}15\,Re^{0{,}687}\right) + \frac{0{,}42}{1 + 4{,}25\cdot 10^4\,Re^{-1{,}16}}.$$

In [ ]:
G = 9.81

def cd_kugla(Re):
    # Clift-Gauvin korelacija za glatku kuglu (Re < ~3e5)
    return 24.0/Re*(1 + 0.15*Re**0.687) + 0.42/(1 + 4.25e4*Re**(-1.16))

def otpor_kugle(v, D_mm, nu_e6, rho):
    D = D_mm/1000.0
    nu = nu_e6*1e-6
    Re = v*D/nu
    Cd = cd_kugla(Re)
    A = np.pi*D**2/4
    F_D = Cd*0.5*rho*v**2*A
    return {'Re': Re, 'Cd': Cd, 'F_D': F_D, 'A': A}

## Interaktivni prikaz

Klizačima se biraju brzina, promjer kugle te viskoznost i gustoća fluida. Prikaz crta univerzalnu krivulju $C_d(Re)$ i označava trenutnu radnu točku; ista krivulja vrijedi za sve veličine i fluide.

In [ ]:
def otpor_prikaz(v, D_mm, nu_e6, rho):
    r = otpor_kugle(v, D_mm, nu_e6, rho)
    Re_k = np.logspace(-1, 5.5, 400)
    Cd_k = cd_kugla(Re_k)

    fig, ax = plt.subplots(figsize=(9, 5.5))
    ax.loglog(Re_k, Cd_k, color='#1565c0', lw=2,
               label='$C_d(Re)$ glatka kugla')
    mask = Re_k < 2
    ax.loglog(Re_k[mask], 24/Re_k[mask], '--', color='#7a8a96',
               lw=1.4, label='Stokes  $C_d = 24/Re$')
    ax.axvspan(2e5, 5e5, color='#f5e6e0', alpha=0.7)
    ax.text(2.4e5, 0.85, 'otporna\nkriza', color='#b7600c',
             fontsize=9, ha='left')
    ax.plot(r['Re'], r['Cd'], 'o', color='#c0392b', ms=11,
             label='radna točka')
    ax.annotate(f"Re = {r['Re']:.3g}\n$C_d$ = {r['Cd']:.3g}",
                 xy=(r['Re'], r['Cd']),
                 xytext=(r['Re']*2.2, r['Cd']*2.8),
                 color='#c0392b', fontsize=10,
                 arrowprops=dict(arrowstyle='->', color='#c0392b'))
    ax.set_xlabel('Reynoldsov broj  $Re = vD/\\nu$')
    ax.set_ylabel('koeficijent otpora  $C_d$')
    ax.set_title(f"Sila otpora  $F_D$ = {r['F_D']:.3g} N")
    ax.grid(True, which='both', ls=':', alpha=0.5)
    ax.legend(loc='upper right', fontsize=9)
    plt.tight_layout()
    plt.show()


interact(
    otpor_prikaz,
    v=FloatSlider(min=0.5, max=60, step=0.5, value=30,
                   description='$v$ (m/s)',
                   layout=Layout(width='420px')),
    D_mm=FloatSlider(min=2, max=200, step=2, value=20,
                      description='$D$ (mm)',
                      layout=Layout(width='420px')),
    nu_e6=FloatSlider(min=1, max=50, step=1, value=15,
                       description='ν (1e-6 m²/s)',
                       layout=Layout(width='420px')),
    rho=FloatSlider(min=1.0, max=1200, step=1, value=1.2,
                     description='ρ (kg/m³)',
                     layout=Layout(width='420px'))
);

## Pitanja za istraživanje

1. **Ista krivulja, razni fluidi.** Namjesti vodu ($\nu = 1$, $\rho = 1000$) i zrak ($\nu = 15$, $\rho = 1{,}2$) tako da postigneš isti $Re$. Zašto je $C_d$ jednak iako su sile otpora posve različite?

2. **Stokesov režim.** Spusti $Re$ ispod $\approx 1$ (mala kugla, velika viskoznost). Poklapa li se krivulja sa $C_d = 24/Re$? Što to znači za taloženje sitnih čestica?

3. **Otporna kriza.** Približi radnu točku području $Re \approx 2\cdot 10^5$. Zašto stvarni $C_d$ tu naglo pada, a korelacija to više ne opisuje?

4. **Skaliranje sile.** Pri fiksnom $C_d$, kako $F_D$ ovisi o $v$ i o $D$? Zašto udvostručenje brzine daje četverostruku silu?

## Veza s teorijom poglavlja

Ovaj prikaz materijalizira središnji rezultat poglavlja U14: dimenzijska analiza svodi problem s pet veličina na funkciju jedne bezdimenzijske varijable, $C_d = f(Re)$. Jedna izmjerena krivulja zato vrijedi za sve veličine i fluide — golema ušteda u odnosu na mjerenje svake kombinacije posebno. Ista logika stoji iza koeficijenta trenja $\lambda(Re, \varepsilon/D)$ i koeficijenta tlaka $C_p$.